# 🔍 Debug Articles 32 & 33 - Chunk Analysis

This notebook helps analyze why Articles 32 and 33 are not being recognized in your CRR RAG chat system.

## Analysis Steps:
1. Load pickle files
2. Inspect chunk structure
3. Search for Articles 32 & 33
4. Analyze metadata and content
5. Compare with working articles
6. Identify root cause

## 1. Import Required Libraries

In [1]:
import pickle
import os
from pathlib import Path
import re
import pandas as pd
from collections import Counter
import json

print("✅ Libraries imported successfully!")

✅ Libraries imported successfully!


## 2. Find and Load PKL Files

In [2]:
# Find all pickle files in current directory
pkl_files = list(Path('.').glob('*.pkl'))

print(f"📁 Found {len(pkl_files)} pickle file(s):")
for pf in pkl_files:
    size_mb = pf.stat().st_size / (1024 * 1024)
    print(f"   - {pf.name} ({size_mb:.2f} MB)")

if not pkl_files:
    print("⚠️  No .pkl files found!")
    print("💡 Make sure you're in the correct directory or specify the path")

📁 Found 2 pickle file(s):
   - legal_chunks_summary_20251118.pkl (0.01 MB)
   - legal_chunks_docling_20251118.pkl (2.22 MB)


In [5]:
# Load the first pickle file (or specify which one)
if pkl_files:
    pkl_file = pkl_files[1]  # Change index if you have multiple files
    print(f"📂 Loading: {pkl_file}")
    
    with open(pkl_file, 'rb') as f:
        data = pickle.load(f)
    
    # Determine structure
    if isinstance(data, list):
        chunks = data
    elif isinstance(data, dict):
        chunks = data.get('chunks', data.get('documents', data.get('data', [])))
    else:
        chunks = [data]
    
    print(f"✅ Loaded {len(chunks)} chunks")
    print(f"📊 Data type: {type(chunks)}")
    print(f"📊 First chunk type: {type(chunks[0]) if chunks else 'N/A'}")
else:
    chunks = []
    print("❌ No data loaded")

📂 Loading: legal_chunks_docling_20251118.pkl
✅ Loaded 1618 chunks
📊 Data type: <class 'list'>
📊 First chunk type: <class 'langchain_core.documents.base.Document'>
✅ Loaded 1618 chunks
📊 Data type: <class 'list'>
📊 First chunk type: <class 'langchain_core.documents.base.Document'>


## 3. Inspect Chunk Structure

In [6]:
# Inspect the structure of the first few chunks
if chunks:
    print("🔬 Inspecting first chunk structure...")
    print("="*80)
    
    first_chunk = chunks[0]
    
    # Show attributes
    if hasattr(first_chunk, '__dict__'):
        print(f"📋 Attributes:")
        for key, value in first_chunk.__dict__.items():
            value_preview = str(value)[:100] if not isinstance(value, (list, dict)) else type(value).__name__
            print(f"   - {key}: {type(value).__name__} = {value_preview}")
    elif isinstance(first_chunk, dict):
        print(f"📋 Dictionary keys:")
        for key in first_chunk.keys():
            value_preview = str(first_chunk[key])[:100] if not isinstance(first_chunk[key], (list, dict)) else type(first_chunk[key]).__name__
            print(f"   - {key}: {type(first_chunk[key]).__name__} = {value_preview}")
    
    # Show metadata if available
    if hasattr(first_chunk, 'metadata'):
        print(f"\n🏷️  Metadata:")
        print(f"   {first_chunk.metadata}")
    elif isinstance(first_chunk, dict) and 'metadata' in first_chunk:
        print(f"\n🏷️  Metadata:")
        print(f"   {first_chunk['metadata']}")

🔬 Inspecting first chunk structure...
📋 Attributes:
   - id: NoneType = None
   - metadata: dict = dict
   - page_content: str = Article  1
Scope
This Regulation lays down uniform rules concerning general prudential requirements 
   - type: str = Document

🏷️  Metadata:
   {'type': 'article', 'article_no': 'Article 1', 'page': 3, 'item_type': 'SectionHeaderItem', 'label': <DocItemLabel.SECTION_HEADER: 'section_header'>}


## 4. Search for Articles 32 & 33

In [7]:
# Search for Articles 32 and 33
def extract_text(chunk):
    """Extract text from chunk regardless of structure"""
    if hasattr(chunk, 'page_content'):
        return chunk.page_content
    elif hasattr(chunk, 'text'):
        return chunk.text
    elif isinstance(chunk, dict):
        return chunk.get('content', chunk.get('text', ''))
    return str(chunk)

def extract_metadata(chunk):
    """Extract metadata from chunk"""
    if hasattr(chunk, 'metadata'):
        return chunk.metadata
    elif isinstance(chunk, dict):
        return chunk.get('metadata', {})
    return {}

# Search for specific articles
target_articles = [32, 33]
article_pattern = re.compile(r'(Article|Članak)\s+(\d+)', re.IGNORECASE)

results = {32: [], 33: []}

print("🔍 Searching for Articles 32 and 33...")
print("="*80)

for i, chunk in enumerate(chunks):
    text = extract_text(chunk)
    
    # Search for article numbers
    for target in target_articles:
        if f"Article {target}" in text or f"Članak {target}" in text:
            results[target].append({
                'index': i,
                'text': text,
                'metadata': extract_metadata(chunk)
            })

# Display results
for article_num in target_articles:
    print(f"\n📌 Article {article_num}:")
    if results[article_num]:
        print(f"   ✅ Found in {len(results[article_num])} chunk(s)")
        for result in results[article_num]:
            print(f"\n   📄 Chunk #{result['index']}:")
            print(f"   Metadata: {result['metadata']}")
            print(f"   Text preview: {result['text'][:300]}...")
            print(f"   Text length: {len(result['text'])} chars")
    else:
        print(f"   ❌ NOT FOUND!")

🔍 Searching for Articles 32 and 33...

📌 Article 32:
   ✅ Found in 21 chunk(s)

   📄 Chunk #80:
   Metadata: {'type': 'article', 'article_no': 'Article 20', 'page': 49, 'item_type': 'SectionHeaderItem', 'label': <DocItemLabel.SECTION_HEADER: 'section_header'>, 'sub_chunk': 1, 'total_sub_chunks': 5, 'original_chunk_tokens': 1683}
   Text preview: Article  20
Joint  decisions  on  prudential  requirements
The competent authorities shall work together, in full consultation:
(a)  in the case of applications for the  permissions referred to in Article 143(1), Article 151(9), Article 283 and Article 325az submitted by  an  EU  parent  institution...
   Text length: 1596 chars

   📄 Chunk #881:
   Metadata: {'type': 'article', 'article_no': 'Article 325', 'page': 466, 'item_type': 'SectionHeaderItem', 'label': <DocItemLabel.SECTION_HEADER: 'section_header'>, 'sub_chunk': 3, 'total_sub_chunks': 5, 'original_chunk_tokens': 1796}
   Text preview: An  institution  may  use  a  combination  of  th

## 5. Analyze All Articles Distribution

In [8]:
# Find all articles in the dataset
all_articles = {}

print("📊 Analyzing article distribution...")

for i, chunk in enumerate(chunks):
    text = extract_text(chunk)
    matches = article_pattern.findall(text)
    
    for article_type, article_num in matches:
        article_num = int(article_num)
        if article_num not in all_articles:
            all_articles[article_num] = 0
        all_articles[article_num] += 1

# Display statistics
if all_articles:
    sorted_articles = sorted(all_articles.items())
    min_article = min(all_articles.keys())
    max_article = max(all_articles.keys())
    
    print(f"\n📈 Article Statistics:")
    print(f"   Range: Article {min_article} - Article {max_article}")
    print(f"   Total unique articles: {len(all_articles)}")
    print(f"   Total chunks with articles: {sum(all_articles.values())}")
    
    # Check for gaps
    all_numbers = set(range(min_article, max_article + 1))
    found_numbers = set(all_articles.keys())
    missing = sorted(all_numbers - found_numbers)
    
    if missing:
        print(f"\n⚠️  Missing articles ({len(missing)} total):")
        print(f"   {missing[:20]}{'...' if len(missing) > 20 else ''}")
        
        # Highlight if 32 or 33 are missing
        if 32 in missing:
            print(f"\n   🔴 Article 32 is MISSING from chunks!")
        if 33 in missing:
            print(f"\n   🔴 Article 33 is MISSING from chunks!")
    
    # Show articles near 32 and 33
    print(f"\n📋 Articles around 32-33:")
    for num in range(28, 38):
        count = all_articles.get(num, 0)
        status = "✅" if count > 0 else "❌"
        print(f"   {status} Article {num}: {count} chunks")
else:
    print("❌ No articles found!")

📊 Analyzing article distribution...

📈 Article Statistics:
   Range: Article 1 - Article 521
   Total unique articles: 483
   Total chunks with articles: 3514

⚠️  Missing articles (38 total):
   [17, 100, 155, 165, 167, 186, 187, 188, 202, 217, 225, 240, 241, 362, 363, 364, 365, 366, 367, 368]...

📋 Articles around 32-33:
   ✅ Article 28: 16 chunks
   ✅ Article 29: 9 chunks
   ✅ Article 30: 1 chunks
   ✅ Article 31: 6 chunks
   ✅ Article 32: 4 chunks
   ✅ Article 33: 6 chunks
   ✅ Article 34: 10 chunks
   ✅ Article 35: 4 chunks
   ✅ Article 36: 89 chunks
   ✅ Article 37: 2 chunks


## 6. Check Metadata Fields

In [9]:
# Check if metadata contains article_number field
metadata_with_article_number = []

print("🏷️  Checking metadata for article_number field...")

for i, chunk in enumerate(chunks[:100]):  # Check first 100
    metadata = extract_metadata(chunk)
    if 'article_number' in metadata:
        article_num = metadata['article_number']
        metadata_with_article_number.append({
            'index': i,
            'article_number': article_num,
            'metadata': metadata
        })

if metadata_with_article_number:
    print(f"\n✅ Found {len(metadata_with_article_number)} chunks with article_number in metadata")
    print(f"\nArticle numbers in metadata:")
    article_nums_in_metadata = [m['article_number'] for m in metadata_with_article_number]
    unique_articles = sorted(set(article_nums_in_metadata))
    print(f"   {unique_articles[:20]}{'...' if len(unique_articles) > 20 else ''}")
    
    # Check if 32 and 33 are in metadata
    if 32 in unique_articles:
        print(f"\n   ✅ Article 32 found in metadata!")
    else:
        print(f"\n   ❌ Article 32 NOT in metadata!")
    
    if 33 in unique_articles:
        print(f"\n   ✅ Article 33 found in metadata!")
    else:
        print(f"\n   ❌ Article 33 NOT in metadata!")
else:
    print("❌ No chunks have article_number in metadata!")
    print("💡 This might be the problem - metadata not properly set during chunking")

🏷️  Checking metadata for article_number field...
❌ No chunks have article_number in metadata!
💡 This might be the problem - metadata not properly set during chunking


## 7. Deep Dive: Fuzzy Search for "32" and "33"

In [ ]:
# Fuzzy search: look for the numbers "32" and "33" anywhere in text
print("🔎 Fuzzy search for numbers '32' and '33'...")
print("="*80)

fuzzy_results = {32: [], 33: []}

for i, chunk in enumerate(chunks):
    text = extract_text(chunk)
    
    # Look for "32" and "33" as standalone numbers or in context
    if " 32" in text or "32 " in text or "32\n" in text or "\n32" in text:
        fuzzy_results[32].append({
            'index': i,
            'preview': text[:500]
        })
    
    if " 33" in text or "33 " in text or "33\n" in text or "\n33" in text:
        fuzzy_results[33].append({
            'index': i,
            'preview': text[:500]
        })

# Show fuzzy results
for num in [32, 33]:
    print(f"\n📌 Number '{num}' appears in {len(fuzzy_results[num])} chunks:")
    if fuzzy_results[num]:
        for result in fuzzy_results[num][:3]:  # Show first 3
            print(f"\n   Chunk #{result['index']}:")
            print(f"   {result['preview']}...")
            print(f"   {'-'*60}")
    else:
        print(f"   ❌ Not found even in fuzzy search!")

## 8. Summary & Diagnosis

Based on the analysis above, determine why Articles 32 & 33 are not recognized.

In [13]:
# Generate diagnosis report
print("🔍 DIAGNOSIS REPORT")
print("="*80)

# Check 1: Are articles in chunks?
articles_32_found = len(results[32]) > 0
articles_33_found = len(results[33]) > 0

print(f"\n1️⃣  Articles in chunks:")
print(f"   Article 32: {'✅ Found' if articles_32_found else '❌ NOT FOUND'}")
print(f"   Article 33: {'✅ Found' if articles_33_found else '❌ NOT FOUND'}")

# Check 2: Are they in metadata?
if metadata_with_article_number:
    article_nums = [m['article_number'] for m in metadata_with_article_number]
    articles_32_in_metadata = 32 in article_nums
    articles_33_in_metadata = 33 in article_nums
    
    print(f"\n2️⃣  Articles in metadata:")
    print(f"   Article 32: {'✅ Found' if articles_32_in_metadata else '❌ NOT FOUND'}")
    print(f"   Article 33: {'✅ Found' if articles_33_in_metadata else '❌ NOT FOUND'}")
else:
    print(f"\n2️⃣  Articles in metadata: ❌ NO METADATA WITH article_number")

# Possible issues
print(f"\n💡 Possible Issues:")
if not articles_32_found and not articles_33_found:
    print(f"   🔴 Articles 32 & 33 are MISSING from the PDF/chunks")
    print(f"      → Check source PDF for these articles")
    print(f"      → Check if these pages were skipped during processing")
elif articles_32_found and not articles_32_in_metadata:
    print(f"   🟡 Articles found in text but NOT in metadata")
    print(f"      → Enhancement function not detecting them")
    print(f"      → Check article pattern matching in enhancement")
elif len(fuzzy_results[32]) == 0 or len(fuzzy_results[33]) == 0:
    print(f"   🔴 Numbers not even found in fuzzy search")
    print(f"      → Definitely missing from source data")
else:
    print(f"   ✅ Articles appear to be present")
    print(f"      → Issue might be in vector store or query process")

print(f"\n📋 Next Steps:")
print(f"   1. Check the source PDF for Articles 32 & 33")
print(f"   2. Verify DoclingReader loaded all pages")
print(f"   3. Check enhancement function pattern matching")
print(f"   4. Query vector store directly for these articles")

🔍 DIAGNOSIS REPORT

1️⃣  Articles in chunks:
   Article 32: ✅ Found
   Article 33: ✅ Found

2️⃣  Articles in metadata: ❌ NO METADATA WITH article_number

💡 Possible Issues:


NameError: name 'articles_32_in_metadata' is not defined